### Step4: attribute analysis and model building
The goal is to extract attributes from step3 results, and analyze correlations between them and the result.
The first part in this notebook contains functions used for attribute extraction.
The second part contains part for attribute analysis, we will conduct the analysis for each exprimental set.

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os

In [ ]:
### Part 1: Attribute Extraction Functions 

### function to find the position of a substring in a string

def find_substring_location(text: any, substring: any) -> tuple[int, float]:
    # — Coerce NaN / non‑str to '' so re.sub never sees a float
    if not isinstance(text, str):
        text = '' if pd.isna(text) else str(text)
    if not isinstance(substring, str):
        substring = '' if pd.isna(substring) else str(substring)

    # — Strip ALL whitespace/newlines
    text_clean      = re.sub(r'\s+', '', text)
    substring_clean = re.sub(r'\s+', '', substring)

    # — If either is empty, no valid match
    if not text_clean or not substring_clean:
        return -1, 0.0

    # — Find all match start‑indices
    matches = [m.start() for m in re.finditer(re.escape(substring_clean), text_clean)]
    first_match = matches[0] if matches else -1

    # — Compute percentage (0.0 if no match)
    pct = (first_match / len(text_clean) * 100.0) if first_match != -1 else 0.0

    return first_match, pct

### function to count the number of lines of code

def count_loc(code: str) -> int:
    lines = code.split("\n")
    non_empty_lines = [line for line in lines if line.strip()]
    return len(non_empty_lines)


In [ ]:
### Read in a csv file, to evaluate the attributes of the code, and determine the procedure to use here.

df1 = pd.read_csv("/Users/anonymous_user/Downloads/experiment_results/exp4_attribute_analysis_results/step4_result_processing/step4_result/sc7b_50.csv")
df1.head()

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import mannwhitneyu
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    classification_report, confusion_matrix
)

def analyze_bleu_model(
    df: pd.DataFrame,
    dv_col: str,
    threshold: float,
    indep_cols = ['loc','percentage_pos','copies'],
    test_size: float = 0.3,
    random_state: int = 42
):
    """
    1) Binarize df[dv_col] > threshold  → new column 'bleu'
    2) Print Pearson corr matrix & VIFs among indep_cols
    3) Mann–Whitney U + medians for each indep
    4) Cliff’s δ effect size
    5) Logistic regression (statsmodels) summary + odds‑ratios + pseudo‑R²
    6) Logistic regression (sklearn) accuracy, AUC, classification report,
       confusion matrix, and learned coefs.

    Returns nothing (prints all results).
    """

    df = df.copy()
    # 1) Binarize
    df['bleu'] = (df[dv_col] > threshold).astype(int)
    print(f"\n>> Thresholding `{dv_col}` > {threshold!r} → 'bleu' flag\n")

    # 2) Correlation & VIF
    indep = df[indep_cols]
    print("== Pearson Correlation Matrix ==\n", indep.corr(), "\n")
    vif = pd.DataFrame({
        'feature': indep_cols,
        'VIF': [variance_inflation_factor(indep.values, i)
                for i in range(len(indep_cols))]
    })
    print("== Variance Inflation Factors ==\n", vif, "\n")

    # 3) Mann–Whitney U
    print("== Mann–Whitney U Tests ==")
    mw_results = []
    for col in indep_cols:
        g0 = df.loc[df['bleu']==0, col]
        g1 = df.loc[df['bleu']==1, col]
        U, p = mannwhitneyu(g0, g1, alternative='two-sided')
        mw_results.append({
            'feature': col,
            'U_statistic': U,
            'p_value': p,
            'median_0': g0.median(),
            'median_1': g1.median()
        })
    print(pd.DataFrame(mw_results).set_index('feature'), "\n")

    # 4) Cliff’s delta
    def cliffs_delta(x, y):
        m, n = len(x), len(y)
        more = sum(1 for xi in x for yi in y if xi > yi)
        less = sum(1 for xi in x for yi in y if xi < yi)
        return (more - less) / (m * n)

    print("== Cliff’s δ Effect Sizes ==")
    for col in indep_cols:
        g0 = df.loc[df['bleu']==0, col].to_numpy()
        g1 = df.loc[df['bleu']==1, col].to_numpy()
        δ = cliffs_delta(g0, g1)
        size = ('negligible' if abs(δ)<0.147 else
                'small' if abs(δ)<0.33 else
                'medium' if abs(δ)<0.474 else
                'large')
        print(f"{col:15s} δ = {δ:.3f} ({size})")
    print()

    # 5) Logistic (statsmodels)
    X = sm.add_constant(df[indep_cols])
    y = df['bleu']
    logit = sm.Logit(y, X).fit(disp=False)
    print("== Logit Regression (statsmodels) ==\n")
    print(logit.summary())
    print("\nOdds ratios:\n", np.exp(logit.params))
    print("\nPseudo R²:", logit.prsquared, "\n")

    # 6) Logistic (sklearn)
    X_train, X_test, y_train, y_test = train_test_split(
        df[indep_cols], y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )
    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(X_train, y_train)

    y_pred  = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:,1]

    print("== Logistic Regression (sklearn) Performance ==\n")
    print("Accuracy:       ", accuracy_score(y_test, y_pred))
    print("ROC AUC:        ", roc_auc_score(y_test, y_proba))
    print("\nClassification report:\n",
          classification_report(y_test, y_pred, zero_division=0))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nIntercept:", clf.intercept_[0])
    print("Coefs:    ", dict(zip(indep_cols, clf.coef_[0])))
    print()

# Example usage:
# analyze_bleu_model(df_sc_7b_150, dv_col='code_only_bleu_150', threshold=0.98)


In [ ]:
df_llama_150 = pd.read_csv("/Users/anonymous_user/Downloads/experiment_results/exp4_attribute_analysis_results/step4_result_processing/step4_result/llama_150.csv")
df_llama_150.head()

In [ ]:
analyze_bleu_model(df_llama_150, dv_col = "code_only_bleu_150", threshold = 0.85)

In [ ]:
analyze_bleu_model(df_sc_7b_150, dv_col='code_only_bleu_150', threshold=0.99)

In [ ]:
### Since the result is pretty trash, we are going to rebalance the dataset, and see if we can get a better result.
### Here we will be using the sc7b_150 input, which has the best performance overall.
df_sc_7b_150 = pd.read_csv("/Users/anonymous_user/Downloads/experiment_results/exp4_attribute_analysis_results/step4_result_processing/step4_result/sc7b_150.csv")

In [ ]:
### compare datasize when the threshold is 0.85
df_sc_7b_150['bleu'] = (df_sc_7b_150['code_only_bleu_150'] > 0.85).astype(int)
print(df_sc_7b_150['bleu'].value_counts())

In [ ]:
analyze_bleu_model(df_sc_7b_150, dv_col='code_only_bleu_150', threshold=0.95)

In [ ]:
### After carefully review, decide to handling the imblance issue by not handling it.
### This kind of imblance reflects the nature of the dataset, hence we are leaving the dataset as it is, but we will be using two bleu score thresholds.T

threshold1 = 0.85
threshold2 = 0.7
analyze_bleu_model(df_sc_7b_150, dv_col='code_only_bleu_150', threshold=threshold1)

In [ ]:
analyze_bleu_model(df_sc_7b_150, dv_col='code_only_bleu_150', threshold = threshold2)